# Neural Network Model for Policy Impact Prediction

This notebook implements a Neural Network to classify the impact of civic policies based on their descriptions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

## 1. Load Dataset

In [ ]:
# Load the collected dataset
df = pd.read_csv("../data/raw/collected dataset.csv")
print("Dataset Shape:", df.shape)
df.head()

## 2. Preprocessing

In [ ]:
# Vectorize policy text using TF-IDF
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X = vectorizer.fit_transform(df['policy_text']).toarray()

# Encode target labels (Low Impact vs High Impact)
le = LabelEncoder()
y = le.fit_transform(df['impact_label'])

print("Feature Matrix Shape:", X.shape)
print("Classes:", le.classes_)

In [ ]:
# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training samples:", X_train.shape)
print("Testing samples:", X_test.shape)

## 3. Build Neural Network Model

In [ ]:
model = Sequential([
    Dense(512, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

## 4. Train Model

In [ ]:
history = model.fit(
    X_train, y_train, 
    epochs=10, 
    batch_size=32, 
    validation_split=0.1,
    verbose=1
)

## 5. Evaluation

In [ ]:
# Evaluate on test set
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {accuracy:.4f}")

# Predictions
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int)

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
# Plot Training History
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Accuracy History')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss History')
plt.legend()
plt.show()

## 6. Real-time Prediction

In [ ]:
def predict_impact(text):
    vec = vectorizer.transform([text]).toarray()
    prob = model.predict(vec)[0][0]
    prediction = "High Impact" if prob > 0.5 else "Low Impact"
    print(f"Policy: {text}")
    print(f"Predicted Impact: {prediction} (Confidence: {prob if prob > 0.5 else 1-prob:.2f})")

test_policy = "The government introduces major institution funding reform"
predict_impact(test_policy)